In [4]:
import pandas as pd
import numpy as np

df = pd.read_csv("marketing_data_cleaned.csv")

In [5]:
df["Dt_Customer"] = pd.to_datetime(
    df["Dt_Customer"],
    errors="coerce"
)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2240 entries, 0 to 2239
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   ID                        2240 non-null   int64         
 1   Year_Birth                2240 non-null   int64         
 2   Education                 2240 non-null   object        
 3   Marital_Status            2240 non-null   object        
 4   Kidhome                   2240 non-null   int64         
 5   Teenhome                  2240 non-null   int64         
 6   Dt_Customer               2240 non-null   datetime64[ns]
 7   Recency                   2240 non-null   int64         
 8   MntWines                  2240 non-null   int64         
 9   MntFruits                 2240 non-null   int64         
 10  MntMeatProducts           2240 non-null   int64         
 11  MntFishProducts           2240 non-null   int64         
 12  MntSweetProducts    

In [7]:
df["Income_Clean"] = df["Income"]

df.loc[df["Income_Clean"] > 200000, "Income_Clean"] = np.nan

In [8]:
#KPI 1 — Total Spending

In [9]:
spending_cols = [
    "MntWines",
    "MntFruits",
    "MntMeatProducts",
    "MntFishProducts",
    "MntSweetProducts",
    "MntGoldProds"
]

df["Total_Spending"] = df[spending_cols].sum(axis=1)

In [17]:
#KPI 2 — Total Purchases

In [18]:
purchase_cols = [
    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases"
]

df["Total_Purchases"] = df[purchase_cols].sum(axis=1)

In [19]:
# KPI 3 — Total Campaign Responses

In [20]:
campaign_cols = [
    "AcceptedCmp1",
    "AcceptedCmp2",
    "AcceptedCmp3",
    "AcceptedCmp4",
    "AcceptedCmp5"
]

In [21]:
df["Total_Campaign_Responses"] = df[campaign_cols].sum(axis=1)

In [22]:
#KPI 4 — Campaign Engagement Level

In [23]:
df["Campaign_Engagement"] = pd.cut(
    df["Total_Campaign_Responses"],
    bins=[-1, 0, 1, 2, 5],
    labels=[
        "No Response",
        "Low Engagement",
        "Medium Engagement",
        "High Engagement"
    ]
)

In [27]:
df["Campaign_Engagement"].value_counts()

Campaign_Engagement
No Response          1777
Low Engagement        325
Medium Engagement      83
High Engagement        55
Name: count, dtype: int64

In [24]:
#KPI 5 — Customer Value Segment

In [25]:
df["Customer_Segment"] = pd.qcut(
    df["Total_Spending"],
    q=3,
    labels=["Low Value", "Medium Value", "High Value"]
)

In [26]:
df["Customer_Segment"].value_counts()

Customer_Segment
Low Value       748
High Value      747
Medium Value    745
Name: count, dtype: int64

In [28]:
#KPI 6 — Age Group

In [30]:
bins = [0, 25, 35, 45, 55, 65, 100]

labels = [
    "Under 25",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65+"
]

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=bins,
    labels=labels,
    right=False
)

In [31]:
df["Age_Group"] .value_counts()

Age_Group
35-44       740
45-54       506
55-64       460
25-34       363
65+         107
Under 25     61
Name: count, dtype: int64

In [32]:
#KPI 7 — Preferred Purchase Channel

In [33]:
channel_cols = [
    "NumWebPurchases",
    "NumCatalogPurchases",
    "NumStorePurchases"
]

channel_names = {
    "NumWebPurchases": "Web",
    "NumCatalogPurchases": "Catalog",
    "NumStorePurchases": "Store"
}

df["Preferred_Channel"] = (
    df[channel_cols]
    .idxmax(axis=1)
    .map(channel_names)
)

In [34]:
df["Preferred_Channel"].value_counts()

Preferred_Channel
Store      1478
Web         602
Catalog     160
Name: count, dtype: int64

In [35]:
#KPI 8 — Total Household Children

In [36]:
df["Total_Children"] = (
    df["Kidhome"] + df["Teenhome"]
)

In [37]:
df["Total_Children"].value_counts().sort_index()

Total_Children
0     638
1    1128
2     421
3      53
Name: count, dtype: int64

In [38]:
#KPI 9 — Household Segment

In [39]:
df["Household_Segment"] = np.select(
    [
        df["Total_Children"] == 0,
        df["Total_Children"] == 1,
        df["Total_Children"] >= 2
    ],
    [
        "No Children",
        "One Child",
        "Two or More Children"
    ],
    default="Unknown"
)

In [41]:
df['Household_Segment'].value_counts()

Household_Segment
One Child               1128
No Children              638
Two or More Children     474
Name: count, dtype: int64

In [42]:
#KPI 10 — Deal Purchase Ratio

In [43]:
df["Deal_Purchase_Ratio"] = np.where(
    df["Total_Purchases"] > 0,
    df["NumDealsPurchases"] / df["Total_Purchases"],
    0
)

In [44]:
df["Deal_Purchase_Rate"] = (
    df["Deal_Purchase_Ratio"] * 100
)

In [47]:
#KPI 11 — Web Purchase Conversion Proxy

In [48]:
df["Web_Purchase_Visit_Ratio"] = np.where(
    df["NumWebVisitsMonth"] > 0,
    df["NumWebPurchases"] / df["NumWebVisitsMonth"],
    np.nan
)

In [49]:
#KPI 12 — Recency Segment

In [50]:
df["Recency_Segment"] = pd.cut(
    df["Recency"],
    bins=[-1, 30, 60, 90, 120, float("inf")],
    labels=[
        "0-30 Days",
        "31-60 Days",
        "61-90 Days",
        "91-120 Days",
        "120+ Days"
    ]
)

In [51]:
#KPI 13 — Campaign Responder Flag

In [52]:
df["Response_Status"] = df["Response"].map({
    0: "Non-Responder",
    1: "Responder"
})

In [53]:
df["Response_Status"].value_counts()

Response_Status
Non-Responder    1906
Responder         334
Name: count, dtype: int64

In [54]:
kpi_summary = {
    "Total Customers": df["ID"].nunique(),
    "Total Spending": df["Total_Spending"].sum(),
    "Total Purchases": df["Total_Purchases"].sum(),
    "Average Customer Spending": df["Total_Spending"].mean(),
    "Average Purchases per Customer": df["Total_Purchases"].mean(),
    "Campaign Response Rate": df["Response"].mean() * 100
}

kpi_summary

{'Total Customers': 2240,
 'Total Spending': np.int64(1356988),
 'Total Purchases': np.int64(28083),
 'Average Customer Spending': np.float64(605.7982142857143),
 'Average Purchases per Customer': np.float64(12.537053571428572),
 'Campaign Response Rate': np.float64(14.910714285714285)}

In [55]:
segment_summary = (
    df.groupby("Customer_Segment", observed=False)
      .agg(
          Customers=("ID", "count"),
          Total_Spending=("Total_Spending", "sum"),
          Avg_Spending=("Total_Spending", "mean"),
          Avg_Purchases=("Total_Purchases", "mean"),
          Campaign_Response_Rate=("Response", "mean")
      )
      .reset_index()
)

In [57]:
segment_summary["Campaign_Response_Rate"] *= 100

In [58]:
segment_summary

,Customer_Segment,Customers,Total_Spending,Avg_Spending,Avg_Purchases,Campaign_Response_Rate
0,Low Value,748,38312,51.219251,4.699198,7.486631
1,Medium Value,745,307667,412.975839,13.044295,12.348993
2,High Value,747,1011009,1353.425703,19.879518,24.899598


In [62]:
df[[
    "ID",
    "Age",
    "Income_Clean",
    "Total_Spending",
    "Total_Purchases",
    "Total_Campaign_Responses",
    "Campaign_Engagement",
    "Customer_Segment",
    "Preferred_Channel",
    "Total_Children",
    "Household_Segment",
    "Deal_Purchase_Rate",
    "Recency_Segment",
    "Response_Status"
]].head(10)

,ID,Age,Income_Clean,Total_Spending,Total_Purchases,Total_Campaign_Responses,Campaign_Engagement,Customer_Segment,Preferred_Channel,Total_Children,Household_Segment,Deal_Purchase_Rate,Recency_Segment,Response_Status
0,1826,44.0,84835.0,1190,14,0,No Response,High Value,Store,0,No Children,7.142857,0-30 Days,Responder
1,1,53.0,57091.0,577,17,1,Low Engagement,Medium Value,Web,0,No Children,5.882353,0-30 Days,Responder
2,10476,56.0,67267.0,251,10,0,No Response,Medium Value,Store,1,One Child,10.000000,0-30 Days,Non-Responder
3,1386,47.0,32474.0,11,3,0,No Response,Low Value,Store,2,Two or More Children,33.333333,0-30 Days,Non-Responder
4,5371,25.0,21474.0,91,6,1,Low Engagement,Low Value,Web,1,One Child,33.333333,0-30 Days,Responder
5,7348,56.0,71691.0,1192,16,0,No Response,High Value,Catalog,0,No Children,6.250000,0-30 Days,Responder
6,4073,60.0,63564.0,1215,27,1,Low Engagement,High Value,Web,0,No Children,3.703704,0-30 Days,Responder
7,1991,47.0,44931.0,96,6,0,No Response,Low Value,Store,1,One Child,16.666667,0-30 Days,Non-Responder
8,4047,60.0,65324.0,544,17,0,No Response,Medium Value,Store,1,One Child,17.647059,0-30 Days,Non-Responder
9,9477,60.0,65324.0,544,17,0,No Response,Medium Value,Store,1,One Child,17.647059,0-30 Days,Non-Responder


In [63]:
segment_summary

,Customer_Segment,Customers,Total_Spending,Avg_Spending,Avg_Purchases,Campaign_Response_Rate
0,Low Value,748,38312,51.219251,4.699198,7.486631
1,Medium Value,745,307667,412.975839,13.044295,12.348993
2,High Value,747,1011009,1353.425703,19.879518,24.899598


In [65]:
df.to_csv(
    "marketing_data_enriched.csv",
    index=False
)